# Proyecto Ebooks - SQL 

## Introducción y objetivo

A raíz de la pandemia del COVID-19, los hábitos de las personas cambiaron, al pasar más tiempo en casa, muchas adoptaron actividades diferentes, entre ellas la lectura. Esto creó una oportunidad para compañías que desarrollan productos digitales, como apps de libros. Con el incremento de lectores, este tipo de aplicaciones ganó relevancia y el mercado se volvió más dinámico y competitivo.

Con la intención de diseñar un nuevo producto digital para este público, se realizará un estudio sobre la base de datos de una empresa que compite en este mercado, con el objetivo de entender el comportamiento y las preferencias de los lectores y, a partir de estos hallazgos, formular una propuesta de valor.

### Objetivo del estudio: 

Explorar el catálogo y el comportamiento de los usuarios, usando como base principal las calificaciones y reseñas que han dejado sobre los libros, para generar una propuesta de valor sustentada en datos.


Todas las consultas se resolverán con SQL y los resultados se presentarán en este notebook.


## Conexión a la base de datos

- Importación de librerías
- Creanción de engine
- Test de conexión

In [1]:
#importando driver para conexión

!pip install psycopg2-binary 

In [2]:
# Importación de librerías

import pandas as pd 
from sqlalchemy import create_engine

# Creación de engine

db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [3]:
# Test de Conexion

pd.read_sql('SELECT 1 AS test_connection;',con=engine)

,test_connection
0,1


## Definición de funcion para ejecución de queries 

In [4]:
## Definición de funcion 

def run_query (query):
    return pd.read_sql(query,con=engine)

## Exploración de tablas

A continuación se ejecutaran los primeros queries exploratorios para:

- Tamaño de tablas.
- Revisar estructura y tipos de columnas.
- Detectar valores nulos y llaves.
- Visualización inicial de las tablas de datos disponibles.


# Diagrama -  Estructura de los Datos 

![Estructura de los Datos](images/data_structure.png)


### Observacion - Estructura de los Datos

Dado que el enunciado del proyecto proporciona la estructura de las tablas y los tipos de datos de cada columna, no fue necesario realizar una exploración adicional de los tipos de datos.

### Exploración del tamaño de las tablas

In [5]:
# Query Tamaño de las tablas

q_table_sizes = (""" 

SELECT 'books' AS table_name, COUNT(*) AS rows FROM books
UNION ALL 
SELECT 'authors' AS table_name, COUNT(*) AS rows FROM authors
UNION ALL 
SELECT 'publishers' AS table_name, COUNT(*) AS rows FROM publishers
UNION ALL 
SELECT 'ratings' AS table_name, COUNT(*) AS rows FROM ratings
UNION ALL 
SELECT 'reviews' AS table_name, COUNT(*) AS rows FROM reviews
""")

run_query(q_table_sizes)

,table_name,rows
0,books,1000
1,authors,636
2,publishers,340
3,ratings,6456
4,reviews,2793


### Observaciones 

Se obtuvieron los tamaños de las tablas con las que se desarrollará el análisis. Esta información permite dimensionar el volumen de datos disponible y anticipar el tipo de relaciones entre las tablas, facilitando la elección de los joins más adecuados y la interpretación correcta del número de observaciones en cada análisis. 

### Tabla books

In [6]:
# Query de Visualización

q_visual_t1 = """ SELECT * FROM books LIMIT 10 """

run_query(q_visual_t1)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268
5,6,257,1st to Die (Women's Murder Club #1),424,2005-05-20,116
6,7,258,2nd Chance (Women's Murder Club #2),400,2005-05-20,116
7,8,260,4th of July (Women's Murder Club #4),448,2006-06-01,318
8,9,563,A Beautiful Mind,461,2002-02-04,104
9,10,445,A Bend in the Road,341,2005-04-01,116


#### Query de revisión de Datos Nulos - Books

In [7]:
# Query de revisión de Datos Nulos 


q_nulls_in_books = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN book_id IS NULL THEN 1 ELSE 0 END) AS book_id_nulls,
SUM(CASE WHEN author_id IS NULL THEN 1 ELSE 0 END) AS author_id_nulls,
SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END) AS title_nulls,
SUM(CASE WHEN num_pages IS NULL THEN 1 ELSE 0 END) AS num_pages_nulls,
SUM(CASE WHEN publication_date IS NULL THEN 1 ELSE 0 END) AS publication_date_nulls,
SUM(CASE WHEN publisher_id IS NULL THEN 1 ELSE 0 END) AS publisher_id_nulls
FROM books;
""")

run_query(q_nulls_in_books)

,total_rows,book_id_nulls,author_id_nulls,title_nulls,num_pages_nulls,publication_date_nulls,publisher_id_nulls
0,1000,0,0,0,0,0,0


### Observaciones - Tabla books

- La visualizacion inicial permite endender mejor la estructura de los datos 

- Se realizó una revisión de valores nulos en la tabla books, enfocada en las columnas clave para el análisis.
  
- Los resultados muestran que no existen valores nulos en ninguna de las columnas evaluadas, lo que indica que la tabla de libros se encuentra completa y no presenta problemas que puedan afectar los joins o los cálculos futuros.


### Tabla authors

In [8]:
q_exp_t2 = """ SELECT * FROM authors LIMIT 10 """

run_query(q_exp_t2)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd
5,6,Alan Paton
6,7,Albert Camus/Justin O'Brien
7,8,Aldous Huxley
8,9,Aldous Huxley/Christopher Hitchens
9,10,Aleksandr Solzhenitsyn/H.T. Willetts


In [9]:
# Query de revisión de Datos Nulos 


q_nulls_in_authors = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN author_id IS NULL THEN 1 ELSE 0 END) AS author_id_nulls,
SUM(CASE WHEN author IS NULL THEN 1 ELSE 0 END) AS author_nulls
FROM authors;
""")

run_query(q_nulls_in_authors)

,total_rows,author_id_nulls,author_nulls
0,636,0,0


### Observaciones - Tabla authors

- La tabla authors contiene un total de 636 registros, correspondientes a los autores del catálogo. 
- No se identificaron valores nulos ni en la columna author_id ni en el nombre del autor, lo que indica que la información de autores está completa y no presenta problemas. Esto permite realizar joins con la tabla books sin riesgo de pérdida de información.

### Tabla publishers 

In [10]:
q_exp_t3 = """ SELECT * FROM publishers LIMIT 10 """

run_query(q_exp_t3)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company
5,6,Aladdin
6,7,Aladdin Paperbacks
7,8,Albin Michel
8,9,Alfred A. Knopf
9,10,Alfred A. Knopf Books for Young Readers


In [11]:
# Query de revisión de Datos Nulos 

q_nulls_in_publishers = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN publisher_id IS NULL THEN 1 ELSE 0 END) AS publisher_id_nulls,
SUM(CASE WHEN publisher IS NULL THEN 1 ELSE 0 END) AS publisher_nulls
FROM publishers;
""")

run_query(q_nulls_in_publishers)

,total_rows,publisher_id_nulls,publisher_nulls
0,340,0,0


### Observaciones - Tabla publishers

- La tabla publishers cuenta con 340 registros y no presenta valores nulos ni en publisher_id ni en el nombre de la editorial. 
- La ausencia de valores faltantes garantiza la correcta vinculación con la tabla books y asegura que los análisis por editorial puedan realizarse de forma correcta.

### Tabla ratings

In [12]:
q_exp_t4 = """ SELECT * FROM ratings LIMIT 10 """

run_query(q_exp_t4)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2
5,6,3,johnsonamanda,4
6,7,3,scotttamara,5
7,8,3,lesliegibbs,5
8,9,4,abbottjames,5
9,10,4,valenciaanne,4


In [13]:
# Query de revisión de Datos Nulos 

q_nulls_in_ratings = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN rating_id IS NULL THEN 1 ELSE 0 END) AS rating_id_nulls,
SUM(CASE WHEN book_id IS NULL THEN 1 ELSE 0 END) AS book_id_nulls,
SUM(CASE WHEN username IS NULL THEN 1 ELSE 0 END) AS username_nulls,
SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS rating_nulls
FROM ratings;
""")

run_query(q_nulls_in_ratings)

,total_rows,rating_id_nulls,book_id_nulls,username_nulls,rating_nulls
0,6456,0,0,0,0


### Observaciones - Tabla ratings

- La tabla ratings contiene 6,456 registros, lo que refleja una alta actividad de calificación por parte de los usuarios.
- No se encontraron valores nulos en las columnas clave (rating_id, book_id, username y rating), lo que indica que todas las calificaciones están correctamente asociadas a un libro y a un usuario.
- Esta estructura permite calcular métricas agregadas, como promedios y conteos, sin necesidad de procesos adicionales por datos faltantes.

### Tabla reviews

In [14]:
q_exp_t5 = """ SELECT * FROM reviews LIMIT 10 """

run_query(q_exp_t5)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...
5,6,3,lesliegibbs,Analysis no several cause international.
6,7,4,valenciaanne,One there cost another. Say type save. With pe...
7,8,4,abbottjames,Within enough mother. There at system full rec...
8,9,5,npowers,Thank now focus realize economy focus fly. Ite...
9,10,5,staylor,Game push lot reduce where remember. Including...


In [15]:
# Query de revisión de Datos Nulos 

q_nulls_in_reviews = (""" 
SELECT 
COUNT(*) AS total_rows,
SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS review_id_nulls,
SUM(CASE WHEN book_id IS NULL THEN 1 ELSE 0 END) AS book_id_nulls,
SUM(CASE WHEN username IS NULL THEN 1 ELSE 0 END) AS username_nulls,
SUM(CASE WHEN text IS NULL THEN 1 ELSE 0 END) AS text_nulls
FROM reviews;
""")

run_query(q_nulls_in_reviews)

,total_rows,review_id_nulls,book_id_nulls,username_nulls,text_nulls
0,2793,0,0,0,0


### Observaciones - Tabla reviews

- La tabla reviews presenta un total de 2,793 registros y no contiene valores nulos en las columnas evaluadas (review_id, book_id, username y text).
- Esto indica que todas las reseñas están completas y correctamente asociadas a libros y usuarios.
- Se detectó una cantidad menor de registros en comparación con la tabla ratings, esto sugiere que no todos los usuarios que califican un libro dejan una reseña de texto.

## Conclusión - Exploracion inicial

En general, no se detectaron valores nulos o faltantes en las tablas analizadas y se identificó que la estructura e integridad de los datos es adecuada, ordenada y bien documentada. Esto permite realizar los joins y análisis necesarios sin riesgo de pérdida de información, proporcionando una base sólida para continuar con el desarrollo del proyecto.

# Estudio del comportamiento de los usuarios 

A continuación se presentan las consultas SQL diseñadas para responder las preguntas de negocio del proyecto y analizar el comportamiento y las preferencias de los usuarios.
.


## Número de libros publicados después del 1 de enero de 2000

In [16]:
# Numero de libros publicados despues de 2000-01-01

q_task1 = ("""
SELECT 
COUNT(DISTINCT book_id)
FROM books
WHERE publication_date > '2000-01-01';
""")

run_query(q_task1)

,count
0,819


# Conclusión 

Del total de libros disponibles en el catálogo (1,000), 819 fueron publicados después del 1 de enero del 2000. Esto indica que aproximadamente el 80 % de los libros disponibles corresponden a publicaciones relativamente recientes.

## Número de reseñas de usuarios y la calificación promedio para cada libro



In [17]:
 

q_task2 = ("""

WITH reviews_per_book AS (
SELECT 
book_id,
COUNT(review_id) AS review_count
FROM reviews
GROUP BY book_id
),

ratings_per_book AS (
SELECT
book_id,
AVG(rating) AS avg_rating
FROM ratings 
GROUP BY book_id
)
SELECT 
b.book_id,
b.title,
rpb.review_count,
rapb.avg_rating
FROM books b
LEFT JOIN reviews_per_book rpb ON b.book_id = rpb.book_id
LEFT JOIN ratings_per_book rapb ON b.book_id = rapb.book_id;

""")

run_query(q_task2)

,book_id,title,review_count,avg_rating
0,652,The Body in the Library (Miss Marple #3),2.0,4.500000
1,273,Galápagos,2.0,4.500000
2,51,A Tree Grows in Brooklyn,5.0,4.250000
3,951,Undaunted Courage: The Pioneering First Missio...,2.0,4.000000
4,839,The Prophet,4.0,4.285714
...,...,...,...,...
995,64,Alice in Wonderland,4.0,4.230769
996,55,A Woman of Substance (Emma Harte Saga #1),2.0,5.000000
997,148,Christine,3.0,3.428571
998,790,The Magicians' Guild (Black Magician Trilogy #1),2.0,3.500000


# Conclusión 

Se obtuvo un dataframe con 1,000 filas, lo que equivale al total de libros disponibles en el catálogo. Para cada libro se agregaron las métricas correspondientes al número de reseñas de usuarios y la calificación promedio, lo que permite analizar el nivel de interacción y la percepción general de los lectores por cada libro de la App.

## Editorial que ha publicado el mayor número de libros con más de 50 páginas

In [18]:
 q_task3 = ("""
SELECT 
p.publisher_id,
p.publisher,
COUNT(b.book_id) AS book_count
FROM books b 
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY
p.publisher_id,
p.publisher
ORDER BY book_count DESC
LIMIT 1;
""")

run_query(q_task3)

,publisher_id,publisher,book_count
0,212,Penguin Books,42


## Conclusión

Se identificó que Penguin Books es la editorial con mayor número de libros publicados con más de 50 páginas dentro del catálogo. En total, esta editorial cuenta con 42 libros que cumplen con el criterio, lo que sugiere una alta participación dentro del segmento de libros completos, excluyendo folletos o publicaciones cortas.

Este hallazgo resulta útil para proponer alianzas o estrategias de contenido con editoriales que tienen una presencia importante en el catálogo y que están bien valoradas por los usuarios, con el objetivo de ofrecer una mejor experiencia y material de mayor interés para los lectores.

## Autor que tiene la más alta calificación promedio del libro

In [19]:
q_task4 = ("""

WITH rating_per_book AS (
SELECT 
book_id,
AVG(rating) AS avg_rating,
COUNT(rating_id) AS rating_count
FROM ratings
GROUP BY book_id
HAVING COUNT(rating_id) >= 50 
)

SELECT
a.author,
a.author_id,
AVG(rpb.avg_rating) AS author_avg_rating,
COUNT(DISTINCT b.book_id) AS books_considered
FROM rating_per_book rpb
JOIN books b ON b.book_id = rpb.book_id
JOIN authors a ON a.author_id = b.author_id
GROUP BY a.author,
a.author_id
ORDER BY author_avg_rating DESC
LIMIT 1;
""")

run_query(q_task4)

,author,author_id,author_avg_rating,books_considered
0,J.K. Rowling/Mary GrandPré,236,4.283844,4


## Conclusión

- Se identificó que J.K. Rowling/Mary GrandPré es el autor con la calificación promedio más alta al considerar únicamente libros con al menos 50 calificaciones.
- Su calificación promedio fue de 4.28, basada en 4 libros que cumplieron con el criterio. Esto sugiere una alta valoración por parte de los usuarios en títulos que cuentan con un volumen significativo de evaluaciones.

- Ofrecer contenido de este autor puede generar un mayor engagement entre los lectores y brindar una mejor experiencia, dado que se estaría recomendando contenido ampliamente valorado por los usuarios.


## Número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

In [20]:


q_task5 = ("""
WITH active_users AS (
    SELECT
        username,
        COUNT(DISTINCT book_id) AS books_rated
    FROM ratings
    GROUP BY username
    HAVING COUNT(DISTINCT book_id) > 50
),
reviews_per_user AS (
    SELECT
        username,
        COUNT(review_id) AS reviews_count
    FROM reviews
    GROUP BY username
)
SELECT
    AVG(COALESCE(rpu.reviews_count, 0)) AS avg_reviews_per_user
FROM active_users au
LEFT JOIN reviews_per_user rpu ON au.username = rpu.username;
""")

run_query(q_task5)

,avg_reviews_per_user
0,24.333333


## Conclusiones finales y propuesta de valor

El análisis del catálogo y del comportamiento de los usuarios permitió identificar patrones clave para la toma de decisiones de negocio. 

1. Se observó que la mayoría de los libros disponibles son relativamente recientes, ya que 819 de los 1,000 títulos fueron publicados después del año 2000, lo que refleja una oferta alineada con intereses actuales.

2. El estudio de la interacción de los usuarios mostró que las calificaciones y reseñas permiten identificar libros y autores altamente valorados. En este sentido, Penguin Books se destaca como la editorial con mayor número de libros completos (más de 50 páginas), con 42 libros en el catálogo, lo que la posiciona como un socio estratégico potencial.

3. J.K. Rowling/Mary GrandPré fue identificado como el autor con la calificación promedio más alta (4.28) al considerar únicamente libros con al menos 50 calificaciones, lo que indica una valoración solida por parte de los usuarios en libros con alto nivel de participación.

4. El análisis de los usuarios más activos reveló que, aunque estos califican una gran cantidad de libros, el número promedio de reseñas de texto es relativamente bajo, lo que representa una oportunidad para incentivar una participación más cualitativa.


## Propuesta de valor

Se recomienda fortalecer el sistema de recomendaciones priorizando libros y autores con alta calificación y volumen de evaluaciones, establecer alianzas estratégicas con editoriales y autores clave, e implementar incentivos que fomenten la creación de reseñas escritas. Estas acciones permitirían mejorar la experiencia del usuario, aumentar el engagement y potenciar el valor del producto digital.
